# Chapter 19 - Sampling the Mind: People and the Kemp Hierarchy

Companion notebook for the [probintro textbook](https://josephausterweil.github.io/probintro/intro2/19_sampling_the_mind/). Run the cells in order; each code cell mirrors a block from the chapter.

## Setup

In [ ]:
!pip install genjax -q
print('ready')

### Step 1 — Gibbs the θᵢ (Conjugate)

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
from jax.scipy.special import betaln, gammaln

# Chibany's 12 bento shops: k_i good tonkatsu ratings out of n_i visits.
# The shops genuinely vary -- some great, some mediocre.
K = jnp.array([9., 3., 7., 5., 8., 2., 6., 9., 4., 7., 1., 8.])    # good ratings
N = jnp.array([10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10.])
M = K.shape[0]

def gibbs_theta(key, phi, kappa):
    a = kappa * phi
    b = kappa * (1.0 - phi)
    keys = jr.split(key, M)
    # theta_i | a, b, k_i, n_i ~ Beta(a + k_i, b + n_i - k_i)  -- conjugate, always accept
    return jax.vmap(lambda kk, ki, ni: jr.beta(kk, a + ki, b + ni - ki))(keys, K, N)

theta = gibbs_theta(jr.key(0), phi=0.6, kappa=5.0)
print("one Gibbs draw of theta_i:", [round(float(t), 2) for t in theta])

### Integrating Out θᵢ: the Beta-Binomial Marginal

In [ ]:
def log_betabin(k, n, a, b):
    log_choose = gammaln(n + 1) - gammaln(k + 1) - gammaln(n - k + 1)   # log C(n, k)
    return log_choose + betaln(a + k, b + n - k) - betaln(a, b)         # log BetaBin

# sanity check: BetaBin is a real distribution, so it must sum to 1 over k = 0..n
ks = jnp.arange(11) * 1.0
total = float(jnp.sum(jnp.exp(jax.vmap(lambda kk: log_betabin(kk, 10.0, 3.0, 2.0))(ks))))
print(f"sum over k of BetaBin(k | n=10, a=3, b=2) = {total:.4f}  (must be 1)")

### The Full Sampler

In [ ]:
def log_marg_all(phi, ell):
    kappa = jnp.exp(ell)
    a = kappa * phi
    b = kappa * (1.0 - phi)
    return jnp.sum(jax.vmap(lambda ki, ni: log_betabin(ki, ni, a, b))(K, N))   # product, in logs

def run_sampler(key, n_sweeps, s_phi=0.04, s_ell=0.25, burn=1000):
    def sweep(carry, k):
        phi, ell = carry
        kg, kp, kq, ka = jr.split(k, 4)
        _ = gibbs_theta(kg, phi, jnp.exp(ell))                  # Gibbs the thetas (conjugate)
        phi_p = phi + s_phi * jr.normal(kp)                     # symmetric proposals on (phi, ell)
        ell_p = ell + s_ell * jr.normal(kq)
        outside = (phi_p <= 0.0) | (phi_p >= 1.0)               # flat prior on phi over (0, 1)
        log_ratio = log_marg_all(phi_p, ell_p) - log_marg_all(phi, ell)  # flat priors: marginal ratio only
        accept = (~outside) & (jnp.log(jr.uniform(ka)) < log_ratio)
        phi = jnp.where(accept, phi_p, phi)
        ell = jnp.where(accept, ell_p, ell)
        return (phi, ell), (phi, jnp.exp(ell), accept)
    init = (0.5, jnp.log(5.0))
    _, (phis, kappas, accs) = jax.lax.scan(sweep, init, jr.split(key, n_sweeps))
    return phis[burn:], kappas[burn:], float(jnp.mean(accs))    # drop burn-in

phis, kappas, acc = run_sampler(jr.key(1), 6000)
print(f"MH acceptance rate:            {acc:.2f}")
print(f"posterior mean phi:            {float(jnp.mean(phis)):.3f}")
print(f"posterior median kappa:        {float(jnp.median(kappas)):.1f}")
print(f"predictive P(next rating good) = mean phi = {float(jnp.mean(phis)):.3f}")

### In GenJAX

In [ ]:
from genjax import gen, beta as gbeta

@gen
def theta_post(a, b):
    return gbeta(a, b) @ "theta"        # the conjugate Beta posterior, as a generative draw

def gibbs_theta_genjax(key, phi, kappa):
    a = kappa * phi
    b = kappa * (1.0 - phi)
    keys = jr.split(key, M)
    return jax.vmap(lambda kk, ki, ni: theta_post.simulate(kk, (a + ki, b + ni - ki)).get_retval())(keys, K, N)

draw = gibbs_theta_genjax(jr.key(0), 0.6, 5.0)
print("GenJAX Gibbs draw of theta_i:", [round(float(t), 2) for t in draw])
# the Metropolis step reuses log_marg_all / log_betabin from above -- it scores the closed-form marginal
print(f"marginal log-score at (phi=0.56, kappa=5): {log_marg_all(0.56, jnp.log(5.0)):.2f}")